# **Clustering**
The goal of this section is to divide users into clusters based on their taste.

In [1]:
import pandas as pd

In [2]:
ratings = pd.read_csv('/Users/beatricecitterio/ratings.csv')
movies = pd.read_csv('/Users/beatricecitterio/movies.csv')

In [3]:
ratings = ratings.drop(columns='timestamp')

In [69]:
movie_counts = ratings['movieId'].value_counts().reset_index()
movie_counts.columns = ['movieId', 'count']

In [70]:
movie_counts

,movieId,count
0,318,102929
1,356,100296
2,296,98409
3,2571,93808
4,593,90330
...,...,...
84427,288825,1
84428,288467,1
84429,287221,1
84430,284087,1


In [92]:
genres = ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'Any Genre']

In [95]:
top_movies = {}
for genre in genres:
    if genre == 'Any Genre':
        mov = movies.merge(movie_counts, on = 'movieId').sort_values(by= 'count', ascending = False)
        top_movies[genre] = mov.nlargest(20, 'count')
    else: 
        genre_movies = movies[movies['genres'].str.contains(genre, case=False)]
        genre_movies = genre_movies.merge(movie_counts, on = 'movieId').sort_values(by= 'count', ascending = False)
        top_movies[genre] = genre_movies.nlargest(20, 'count')


In [96]:
top_movies['Any Genre']

,movieId,title,genres,count
314,318,"Shawshank Redemption, The (1994)",Crime|Drama,102929
351,356,Forrest Gump (1994),Comedy|Drama|Romance|War,100296
292,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,98409
2480,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,93808
585,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,90330
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,85010
2867,2959,Fight Club (1999),Action|Crime|Drama|Thriller,77332
475,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,75233
522,527,Schindler's List (1993),Drama|War,73849
4888,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,73122


In [97]:
top_movies['Thriller']

,movieId,title,genres,count
52,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,98409
431,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,93808
110,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,90330
487,2959,Fight Club (1999),Action|Crime|Drama|Thriller,77332
89,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,75233
9,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,67750
8,47,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,63298
111,608,Fargo (1996),Comedy|Crime|Drama|Thriller,58031
2456,79132,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,57931
134,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,57224


## **EXAMPLE**

I am a new user. The system asks me which type of film I want to see, I say 'Thriller'. The system will then ask me to give a rating to the top10 most famous thriller movies.

In [318]:
top10 = top_movies['Thriller'][:10]

My input is the following:

In [116]:
new_rating = [5, 'Not seen', 4.5, 5, 'Not seen', 'Not seen', 5, 'Not seen', 4.5, 'Not seen']

In [325]:
new_rating_dict = {}
for i in range(10):
    new_rating_dict[list(top10.movieId)[i]] = new_rating[i]

In [326]:
new_rating_filtered =  {k: v for k, v in new_rating_dict.items() if v != 'Not seen'}


In [327]:
new_rating_filtered

{296: 5, 593: 4.5, 2959: 5, 47: 5, 79132: 4.5}

In [328]:
filtered_ratings = ratings[ratings['movieId'].isin(new_rating_filtered.keys())]

In [329]:
new_rating_filtered.values()

dict_values([5, 4.5, 5, 5, 4.5])

In [330]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances

In [331]:
pivot_df = filtered_ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)
pivot_df

movieId,47,296,593,2959,79132
userId,,,,,
1,0.0,0.0,3.0,0.0,0.0
2,0.0,1.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0
5,3.0,1.0,3.0,0.0,0.0
7,0.0,5.0,0.0,0.0,0.0
...,...,...,...,...,...
200944,4.5,4.5,4.0,5.0,5.0
200945,4.0,0.5,4.0,5.0,5.0
200946,0.0,5.0,5.0,0.0,0.0


In [332]:
new_user_ratings = pd.DataFrame([new_rating_filtered], index=['new_user'])

In [333]:
dissimilarities = euclidean_distances(pivot_df, new_user_ratings)

In [335]:
most_similar_user = pivot_df.index[np.argmin(dissimilarities)]

print(f'Most similar user ID: {most_similar_user}')

Most similar user ID: 8


In [337]:
user_8 = pivot_df.loc[8]
user_8

movieId
47       5.0
296      4.5
593      5.0
2959     5.0
79132    4.5
Name: 8, dtype: float64

In [340]:
user_ratings = ratings[ratings['userId'] == 8]
thriller_movies = movies[movies['genres'].str.contains('Thriller', case=False)]
user_ratings = user_ratings[user_ratings['movieId'].isin(thriller_movies.movieId)]

In [343]:
user_ratings

,userId,movieId,rating
470,8,32,4.0
471,8,47,5.0
473,8,296,4.5
475,8,593,5.0
476,8,608,5.0
482,8,2712,4.0
484,8,2959,5.0
486,8,4011,3.5
487,8,4226,4.5
489,8,6016,4.5


In [344]:
user_ratings_filtered = user_ratings.sort_values(by = 'rating', ascending=False)
# QUA NON METTERE NECESSARIAMENTE RATING DI 5 BASTA METTERE RATING IN ORDINE DI GRANDEZZA

In [345]:
user_ratings_filtered

,userId,movieId,rating
471,8,47,5.0
475,8,593,5.0
476,8,608,5.0
484,8,2959,5.0
492,8,27773,5.0
473,8,296,4.5
487,8,4226,4.5
489,8,6016,4.5
495,8,44555,4.5
498,8,79132,4.5


In [346]:
suggested_movies = user_ratings_filtered

In [347]:
suggested_movies

,userId,movieId,rating
471,8,47,5.0
475,8,593,5.0
476,8,608,5.0
484,8,2959,5.0
492,8,27773,5.0
473,8,296,4.5
487,8,4226,4.5
489,8,6016,4.5
495,8,44555,4.5
498,8,79132,4.5


In [348]:
suggested_movies = suggested_movies[~suggested_movies['movieId'].isin(new_rating_filtered.keys())]

In [357]:
suggested_movies = suggested_movies.merge(movie_counts).sort_values(by = ['rating', 'count'], ascending=False)

In [358]:
final_suggestions = suggested_movies[:3]
final_suggestions = movies[movies['movieId'].isin(final_suggestions.movieId)].title

In [359]:
final_suggestions

600       Fargo (1996)
4123    Memento (2000)
9338    Old Boy (2003)
Name: title, dtype: object

## **EXAMPLE 2** 